# Unnest the campaigns and viewed items

*Data Analysis in Python · Topic 7.2*

Which campaigns brought sessions in Black Friday week, and how many tagged sessions lost their campaign on the way? And which product categories did people view? The campaign is in each event's `event_params`, and the products are in its `items`.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that downloads the GA4 sample and the cell that creates `records`: one dictionary per event (topic 7.1.'s steps).
2. In the first cell marked ✏️, add a `params` key to every dictionary in `records`: the event's `event_params` as a dictionary, with each parameter's `key` as its key and its `value` as its value. Then create the `events` table: `records` flattened, one row per event.
3. In the second cell marked ✏️, create:
   - the `session_starts` table: the rows of the `events` table whose `event_name` is `session_start`
   - `campaign_sessions`: the number of sessions per value of the `campaign` parameter, as a Series, with the sessions that don't have the parameter counted too
   - `lost_campaigns`: the number of sessions that have a `source` parameter but no `campaign` parameter: their landing page had UTM tags, and the campaign tag got lost
4. In the third cell marked ✏️, create:
   - the `viewed_items` table: one row per item of the `view_item` events, made from `records`, with each event's `event_timestamp`, `user_pseudo_id` and device category as columns
   - `views_by_category`: the number of `viewed_items` rows per `item_category2`, as a Series
5. In the last cell marked ✏️, create:
   - `item_value`: the total of the `viewed_items` table's price times quantity
   - `event_value`: the total of the `view_item` events' `value` parameter, from the `events` table
6. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `events` table has 12,511 rows and 53 columns, including `params.campaign.string_value`. The `session_starts` table has 2,000 rows. `campaign_sessions` has 33 rows, with `NaN` 829 and `2025_us_email_blackfriday` 138, and `lost_campaigns` is `18`. The `viewed_items` table has 1,115 rows, with `event_timestamp`, `user_pseudo_id` and `device.category` columns, and `views_by_category` has 12 categories, with Pants 226 and Swim 162. `item_value` and `event_value` are both about 68,004.85.

## Parameters

In [ ]:
import json
import urllib.request

import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
EVENTS_FILE = "events_ga4_sample.ndjson"

In [ ]:
# Download the GA4 sample to this session's files. Run it, and leave it as it is.
urllib.request.urlretrieve(DATA_URL + EVENTS_FILE, EVENTS_FILE)
print(f"Downloaded {EVENTS_FILE}")

In [ ]:
# One dictionary per event (topic 7.1). Run it, and leave it as it is.
with open(EVENTS_FILE, encoding="utf-8") as file:
    records = [json.loads(line) for line in file]
len(records)

## The parameters as columns

In [ ]:
for record in records:
    record["params"] = {param["key"]: param["value"] for param in record["event_params"]}
events = pd.json_normalize(records)
events.shape

## The campaigns

In [ ]:
session_starts = events[events["event_name"] == "session_start"]
campaign_sessions = session_starts["params.campaign.string_value"].value_counts(dropna=False)
lost_campaigns = (session_starts["params.source.string_value"].notna() & session_starts["params.campaign.string_value"].isna()).sum()
campaign_sessions.head(), lost_campaigns

## The viewed items

In [ ]:
view_records = [record for record in records if record["event_name"] == "view_item"]
viewed_items = pd.json_normalize(view_records, record_path="items", meta=["event_timestamp", "user_pseudo_id", ["device", "category"]])
views_by_category = viewed_items["item_category2"].value_counts()
viewed_items.shape, views_by_category

## Do the items add up?

In [ ]:
item_value = (viewed_items["price"] * viewed_items["quantity"]).sum()
view_events = events[events["event_name"] == "view_item"]
event_value = view_events["params.value.double_value"].sum()
item_value, event_value

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    events = found.get("events")
    if not isinstance(events, pd.DataFrame):
        results.append((False, "events isn't a DataFrame yet. Add params to every record, then flatten: pd.json_normalize(records)."))
    elif "params.campaign.string_value" not in events.columns:
        if "params" in events.columns or "params.campaign" in events.columns:
            results.append((False, "The events table has params as dictionaries, not columns. Flatten records with pd.json_normalize(records), without max_level."))
        else:
            results.append((False, "The events table has no params.campaign.string_value column. In the loop, set record[\"params\"] = {param[\"key\"]: param[\"value\"] for param in record[\"event_params\"]}, then flatten records."))
    elif len(events) != 12511:
        results.append((False, f"The events table should have 12,511 rows, one per event. It has {len(events):,}."))
    else:
        results.append((True, f"The events table has 12,511 events, and {sum(column.startswith('params.') for column in events.columns)} parameter columns."))

    starts = found.get("session_starts")
    campaigns = found.get("campaign_sessions")
    lost = found.get("lost_campaigns")
    if isinstance(campaigns, pd.DataFrame) and campaigns.shape[1] == 1:
        campaigns = campaigns.iloc[:, 0]
    if not isinstance(starts, pd.DataFrame) or len(starts) != 2000:
        results.append((False, "session_starts should be the events table's 2,000 session_start rows: events[events[\"event_name\"] == \"session_start\"]."))
    elif not isinstance(campaigns, pd.Series):
        results.append((False, "campaign_sessions should be a Series: session_starts[\"params.campaign.string_value\"].value_counts(dropna=False)."))
    elif campaigns.sum() == 1171 and not campaigns.index.isna().any():
        results.append((False, "campaign_sessions adds up to 1,171 sessions, not 2,000: value_counts() leaves out the 829 sessions without a campaign parameter. Add dropna=False."))
    elif campaigns.sum() != 2000 or campaigns.get("2025_us_email_blackfriday") != 138:
        results.append((False, f"campaign_sessions should count the session_start rows by params.campaign.string_value, 2,000 in all, with 2025_us_email_blackfriday 138. It adds up to {campaigns.sum():,}."))
    elif campaigns[campaigns.index.isna()].sum() != 829:
        results.append((False, "campaign_sessions should have NaN as one of its values, with 829 sessions: value_counts(dropna=False)."))
    elif not number(lost):
        results.append((False, f"lost_campaigns should be a number, the sessions with a source parameter but no campaign parameter. It is {lost!r}."))
    elif lost == 829:
        results.append((False, "lost_campaigns is 829: every session without a campaign parameter, including the 811 that had no UTM tags at all. Count only the ones with a source parameter: params.source.string_value isn't NaN."))
    elif lost == 811:
        results.append((False, "lost_campaigns is 811: those are the sessions without a source parameter. Count the ones that have a source and no campaign."))
    elif lost != 18:
        results.append((False, f"lost_campaigns should be 18: session_start rows where params.source.string_value isn't NaN and params.campaign.string_value is. It is {lost!r}."))
    else:
        results.append((True, f"The sessions' campaigns: {campaigns.sum() - 829:,} sessions with a campaign, 811 without UTM tags, and 18 that were tagged but lost their campaign."))

    viewed = found.get("viewed_items")
    by_category = found.get("views_by_category")
    if isinstance(by_category, pd.DataFrame) and by_category.shape[1] == 1:
        by_category = by_category.iloc[:, 0]
    if not isinstance(viewed, pd.DataFrame):
        results.append((False, "viewed_items should be a DataFrame: pd.json_normalize() with record_path=\"items\" on the view_item records."))
    elif "items" in viewed.columns:
        results.append((False, "The viewed_items table still has an items column. Make the rows from the list with record_path=\"items\", from the view_item records."))
    elif len(viewed) == 1799:
        results.append((False, "The viewed_items table has 1,799 rows: the items of every event, including add_to_cart, begin_checkout and purchase. Keep only the view_item records first: [record for record in records if record[\"event_name\"] == \"view_item\"]."))
    elif len(viewed) != 1115 or "item_category2" not in viewed.columns:
        results.append((False, f"The viewed_items table should have 1,115 rows, one per viewed item, with the items' columns such as item_category2. It has {len(viewed):,} rows."))
    elif not {"event_timestamp", "user_pseudo_id", "device.category"} <= set(viewed.columns):
        results.append((False, f"The viewed_items table should have the columns event_timestamp, user_pseudo_id and device.category: meta=[\"event_timestamp\", \"user_pseudo_id\", [\"device\", \"category\"]]. It has {list(viewed.columns)}."))
    elif not isinstance(by_category, pd.Series) or by_category.get("Pants") != 226:
        if isinstance(by_category, pd.Series) and by_category.get("Women") == 721:
            results.append((False, "views_by_category counts item_category: Women, Men and Unisex. Count item_category2, the product category."))
        else:
            results.append((False, "views_by_category should be viewed_items[\"item_category2\"].value_counts(): Pants 226, Swim 162."))
    elif len(by_category) != 12 or by_category.sum() != 1115:
        results.append((False, f"views_by_category should have 12 categories and add up to 1,115 views. It has {len(by_category)}, adding up to {by_category.sum():,}."))
    else:
        results.append((True, f"1,115 viewed items, most of them Pants ({by_category['Pants']}) and Swim ({by_category['Swim']})."))

    item_value = found.get("item_value")
    event_value = found.get("event_value")
    if not number(item_value) or not number(event_value):
        results.append((False, f"item_value and event_value should be numbers. They are {item_value!r} and {event_value!r}."))
    elif abs(event_value - 92361.63) < 0.01:
        results.append((False, "event_value is $92,361.63: the value parameter of every event that has one, including add_to_cart and purchase. Total it for the view_item events only."))
    elif abs(item_value - 68004.85) > 0.01:
        results.append((False, f"item_value should be the viewed_items table's price times quantity, totaled: about 68,004.85. It is {item_value:,.2f}."))
    elif abs(event_value - 68004.85) > 0.01:
        results.append((False, f"event_value should be the view_item events' params.value.double_value, totaled: about 68,004.85. It is {event_value:,.2f}."))
    else:
        results.append((True, "The viewed items add up to their events' value: $68,004.85 both ways."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
for record in records:
    record["params"] = {param["key"]: param["value"] for param in record["event_params"]}
events = pd.json_normalize(records)
events.shape
```

```python
session_starts = events[events["event_name"] == "session_start"]
campaign_sessions = session_starts["params.campaign.string_value"].value_counts(dropna=False)
lost_campaigns = (session_starts["params.source.string_value"].notna() & session_starts["params.campaign.string_value"].isna()).sum()
campaign_sessions.head(), lost_campaigns
```

```python
view_records = [record for record in records if record["event_name"] == "view_item"]
viewed_items = pd.json_normalize(view_records, record_path="items", meta=["event_timestamp", "user_pseudo_id", ["device", "category"]])
views_by_category = viewed_items["item_category2"].value_counts()
viewed_items.shape, views_by_category
```

```python
item_value = (viewed_items["price"] * viewed_items["quantity"]).sum()
view_events = events[events["event_name"] == "view_item"]
event_value = view_events["params.value.double_value"].sum()
item_value, event_value
```

</details>